<br>
<p float="right">
  <img src=attachment:nanoHUB_logo_color.png width="25%" height='10%' align="right" /> 
</p>

# 1: LLM Script Generation

### <i>Ethan Holbrook, Juan C. Verduzco, </i>  and <i>Alejandro Strachan </i>
### Materials Engineering, Purdue University <br>

## Overview

This notebook is the first stage of the evaluation pipeline. It sends fixed method descriptions to selected large language models and stores the returned LAMMPS input scripts in a standardized directory structure for downstream analysis.

The prompts, model identifiers, and output paths used here match the conditions of the benchmark reported in this repository. The notebook is intended to generate candidate scripts only; correctness is assessed in later stages through sanitization, parsing, execution, and prompt-specific accuracy checks.

As it is currently setup, it will only generate a single script for each prompt for gpt-4o. Modifying loops where trial is mentioned will allow for multiple scripts for the same prompt/model combo. For more prompt/model combos, edit the prompt_model_map dictionary. This code generates many folder for storing the LAMMPS input files along the pipeline. 

## Tips
1. Found a bug? Email holbrooe@purdue.edu

<br><br><br>

# Libraries

In [36]:
import os
import json
import sys
import shutil
import re

from dotenv import load_dotenv
load_dotenv()

from lark import tree
from lammps_ast.sanitizer import sanitize
from lammps_ast.parser import parse_to_AST

import lammps_ast
print(dir(lammps_ast))
print(lammps_ast.__path__)

import openai
from openai import OpenAI

import numpy as np
import pandas as pd

import subprocess

from colorama import Fore, Style

from importlib.metadata import version
print(version("lammps-ast"))

# import anthropic

['__author__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'error_handler', 'parse_to_AST', 'parser', 'sanitize', 'sanitizer', 'transformer']
['/home/holbrooe/.conda/envs/2022.10-py39/gst/lib/python3.12/site-packages/lammps_ast']
0.1.7


# Functions
## API Text Extraction + LAMMPS Submission

In [37]:
def text_to_file(response_text, prompt_type, model_type, trial, path_to_dirs):
    """
    Extracts the LAMMPS script from the response text and writes it to a file.
    Handles both markdown-style and fallback formats.
    """

    # Improved regex to capture any code inside triple backticks, with optional language specifier
    code_blocks = re.findall(r"```(?:lammps|bash|[\w]*)?\n(.*?)\n```", response_text, re.DOTALL)

    if code_blocks:
        # Take the first code block found (assuming it's the correct one)
        lammps_script = code_blocks[0].strip()
        print('fallback 0')
    else:
        # Fallback: Try extracting from within separators (if regex fails)
        print('fallback 1')
        match = re.search(r"[-=]{10,}\n(.*?)\n[-=]{10,}", response_text, re.DOTALL)
        if match:
            lammps_script = match.group(1).strip()
        else:
            # Fallback #2: Try extracting the longest section with LAMMPS-style keywords
            print('fallback 2')
            lines = response_text.splitlines()
            lammps_lines = [line for line in lines if re.match(r"^\s*(units|atom_style|lattice|region|create|mass|pair_style|pair_coeff|velocity|fix|run|write_)", line)]
            if lammps_lines:
                lammps_script = "\n".join(lammps_lines)
            else:
                raise ValueError("No LAMMPS script found in response.")

    # Construct output directory and filename
    output_dir = os.path.join(path_to_dirs, prompt_type, model_type)
    os.makedirs(output_dir, exist_ok=True)  # Ensure directories exist

    output_filename = os.path.join(output_dir,f'{prompt_type}-{model_type}-T{trial}.in')    
#     output_filename = os.path.join(output_dir, f'lmmp-{model_type}-T{trial}.in')

    # Write to file
    with open(output_filename, "w") as file:
        file.write(lammps_script)

    print(f"LAMMPS script written to {output_filename}")
    
def text_to_file_basic(response_text, prompt_type, model_type, trial, path_to_dirs):
    """
    Extracts the LAMMPS script from the response text and writes it to a file.
    Handles both markdown-style and fallback formats.
    """

    # Construct output directory and filename
    output_dir = os.path.join(path_to_dirs, prompt_type, model_type)
    os.makedirs(output_dir, exist_ok=True)  # Ensure directories exist

    output_filename = os.path.join(output_dir,f'{prompt_type}-{model_type}-T{trial}.in')    
#     output_filename = os.path.join(output_dir, f'lmmp-{model_type}-T{trial}.in')

    # Write to file
    with open(output_filename, "w") as file:
        file.write(response_text)

    print(f"LAMMPS script written to {output_filename}")
    
def write_submission_script(write_dir, job_name, input_file, log_file): #output_file, exec

    
    submit_script=f'''#!/bin/tcsh
#SBATCH -t 00:30:00
#SBATCH -A strachan 
#SBATCH -N 1
#SBATCH --tasks-per-node=1
#SBATCH --job-name={job_name}

set echo

cd $SLURM_SUBMIT_DIR

module purge

module load gcc/12.2.0
module load openmpi/4.1.4
module load intel-mkl/2019.9.304
module load lammps

mpiexec -n 1 lmp -in {input_file} -log {log_file}'''
# mpiexec -n 128 {exec} -in {input_file} -log {log_file} >> {output_file}'''

    with open(os.path.join(write_dir,job_name+'.sh'),'w') as w_file:
        w_file.writelines(submit_script)
    
    
# log_path = os.path.join(write_path,name+'.log')
# input_script = os.path.join(model_dir,script)
# write_submission_script(write_path,name,input_script,log_path)

# API KEYS for Script Generation

In [41]:
openai_key = os.getenv("OPENAI_API")
client = OpenAI(api_key=openai_key)

# anthropic_key = os.getenv("ANTHROPIC_API")
# client2 = anthropic.Anthropic(api_key=anthropic_key)


## Prompts 1,2,3 - System Prompt

In [42]:
prompt1 = (r"""Method Description:

We used molecular dynamics with LAMMPS to equilibrate an Al sample under isobaric, isothermal conditions (NPT ensemble) at 300 K and 1 atm for 500 ps. The initial conditions were obtained by replicating the fcc unit cell 5 times in each direction. We used Nose-Hoover thermostat and barostat with relaxation timescales of 0.1 and 1 ps, respectively. All MD simulations were performed using LAMMPS. Atomic interactions were obtained using an embedded atom model developed by Ercolessi and Adams [1] obtained from OpenKIM.org. [1] EAM alloy potential for Al developed by Ercolessi and Adams (1994) v002. OpenKIM. 2018. doi:10.25950/376e3e7e.""")

In [43]:
prompt2=(r"""Method Description:

We characterized the melting of a bulk Ni sample using molecular dynamics with LAMMPS. The initial condition was obtained by replicating the Ni unit cell 10 times in each direction. Initial velocities were drawn from the Maxwell-Boltzmann distribution at 600 K. The system was heated from 300 K to 2500 K continuously, at a rate of 10 K per ps under isothermal and isobaric conditions at 1 atm. Interactions were described using an embedded atom model developed by Mishin et al. in 1999 [1] obtained from OpenKIM.org. [1] EAM potential (LAMMPS cubic hermite tabulation) for Ni developed by Mishin et al. (1999) v005. OpenKIM; 2018. doi:10.25950/a88dfc37.""")

In [44]:
prompt3=(r"""Method Description: 

We simulate spall failure on Nb single crystals using high-velocity impact simulations using molecular dynamics (MD) with the LAMMPS code. We simulate the impact of a projectile on a target with a relative velocity of 2 km/s. The projectile is obtained by replicating the Nb BCC unit cell 20 times along the [100], [010], and [001]; the target is longer along the shock direction and is obtained by replicating the BCC unit cell 20 times along [100] and [010] and 40 times along [001]. We apply periodic boundary conditions along the directions normal to the impact direction, [001], and free boundaries along [001]. A gap of 1.5 nm initially separates the target and projectile. The system is equilibrated at 300 K for 100 ps using isothermal, isochoric MD. An impact velocity of 2 km/s is added to the thermal velocities to all the atoms in the projectile along [001] in the direction of the target. Adiabatic MD is used to simulate the impact and subsequent expansion. All atomic interactions are described using an EAM potential developed by Fellinger et al. [1] and downloaded from openKIM [2]. [1] Fellinger MR, Park H, Wilkins JW. Force-matched embedded-atom method potential for niobium. Physical Review B. 2010Apr;81(14):144119. doi:10.1103/PhysRevB.81.144119 [2] https://doi.org/10.25950/befb2eea.""")

In [45]:
prompts = [prompt1,prompt2,prompt3]

In [46]:
SYSTEM_PROMPT = (
    "System Prompt:\n"
    "You are an expert in molecular dynamics simulations and LAMMPS scripting. Your task is to generate complete, runnable LAMMPS input scripts based only on the provided method description. "
    "You must follow strict formatting rules and proceed step by step, reasoning through each section logically before writing the final output. "
    "All reasoning should be internal. Do not display intermediate thoughts, summaries, or explanations—only output the final script.\n\n"
    "Input Format:\n"
    "You will receive a user input labeled Method Description. This section contains all the experimental and simulation details.\n\n"
    "Chain-of-Thought Workflow:\n"
    "Parse and interpret the method description carefully and extract only the information explicitly stated. Do not assume any values or simulation parameters beyond what is given.\n"
    "Plan the LAMMPS input script structure, using the following clearly labeled sections:\n"
    "Initialization\n"
    "System Construction\n"
    "Potential\n"
    "Miscellaneous (if needed)\n"
    "Production Run\n\n"
    "Generate each section by:\n"
    "- Determine what LAMMPS commands are required based on the method description.\n"
    "- Use default values only when the method explicitly mentions them or when required by LAMMPS syntax, but still write them explicitly.\n"
    "- Ensure syntax correctness and functional completeness.\n\n"
    "Constraints:\n"
    "- Assume that a potential file named '../../../potentials/prompt1.potential' is available and ready to use.\n"
    "- Be specific with the selection of the pair style for the potential discussed in the method description. "
    "The pair style must match the format and type of the potential file referenced or described in the method description (e.g., eam/alloy, meam, tersoff, reaxff, etc.). "
    "If a citation or filename indicates a specific format, you must infer the correct pair_style accordingly.\n"
    "- Do not include any comments.\n"
    "- Explicitly define all commands, even if using the defaults.\n"
    "- After reasoning through all steps, output a single clean script, correctly structured and ready to run. "
    "The output must be valid LAMMPS syntax and runnable without modification. "
    "Do not include explanations, metadata, or commentary—only output the final script."
)

print(SYSTEM_PROMPT)

System Prompt:
You are an expert in molecular dynamics simulations and LAMMPS scripting. Your task is to generate complete, runnable LAMMPS input scripts based only on the provided method description. You must follow strict formatting rules and proceed step by step, reasoning through each section logically before writing the final output. All reasoning should be internal. Do not display intermediate thoughts, summaries, or explanations—only output the final script.

Input Format:
You will receive a user input labeled Method Description. This section contains all the experimental and simulation details.

Chain-of-Thought Workflow:
Parse and interpret the method description carefully and extract only the information explicitly stated. Do not assume any values or simulation parameters beyond what is given.
Plan the LAMMPS input script structure, using the following clearly labeled sections:
Initialization
System Construction
Potential
Miscellaneous (if needed)
Production Run

Generate eac

# Models

* GPT-4o
* GPT-o3
* GPT-4.1
* Claude 4 opus
* GPT-5

In [55]:
current_dir = os.getcwd()
# parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
parent_dir = os.path.abspath(os.path.join(current_dir))
scripts_dir = os.path.join(parent_dir,"generated_scripts")
scripts_base = os.path.join(parent_dir,"generated_scripts")

print(scripts_dir)

prompt_dirs = ['prompt1','prompt2','prompt3']
model_dirs = ['gpt-4o','gpt-4.1','gpt-o3','claude-opus-4','gpt-5'] #,'gpt-4o-search',]
# model_dirs = [] #,'gpt-4o-search',]
model_names = ['gpt-4o-2024-08-06','gpt-4.1-2025-04-14','o3-2025-04-16','claude-4-opus-20250514','gpt-5-2025-08-07']
# model_names = ['gpt-5-2025-08-07']

/home/holbrooe/LAMMPS-AST/EvaluationPipelineExample/generated_scripts


## Initialization of directory variables

In [56]:
prompt_choice = 1
model_choice = 0

prompt_type = prompt_dirs[prompt_choice-1]
print(prompt_type)
model_type = model_dirs[model_choice]
model_name = model_names[model_choice]
print(model_type,model_name)

trial=0
# output_filename = os.path.join(prompt_type, model_type,f'P_{prompt_type}-M_{model_type}-T_{trial}.in')

print(prompt_dirs)
print(model_dirs)
print(model_names)

prompt_dict = {'prompt1':prompts[0],'prompt2':prompts[1],'prompt3':prompts[2]}

model_dict = {model_dirs[0]:model_names[0],
              model_dirs[1]:model_names[1],
              model_dirs[2]:model_names[2],
              model_dirs[3]:model_names[3],
              model_dirs[4]:model_names[4],
             }

prompt1
gpt-4o gpt-4o-2024-08-06
['prompt1', 'prompt2', 'prompt3']
['gpt-4o', 'gpt-4.1', 'gpt-o3', 'claude-opus-4', 'gpt-5']
['gpt-4o-2024-08-06', 'gpt-4.1-2025-04-14', 'o3-2025-04-16', 'claude-4-opus-20250514', 'gpt-5-2025-08-07']


# Script Generation

Code that calls the LLM APIs to generate the scripts (optionally putting into the folders)

## OpenAI script maker

In [57]:
print(model_dict)
print(prompts)

{'gpt-4o': 'gpt-4o-2024-08-06', 'gpt-4.1': 'gpt-4.1-2025-04-14', 'gpt-o3': 'o3-2025-04-16', 'claude-opus-4': 'claude-4-opus-20250514', 'gpt-5': 'gpt-5-2025-08-07'}
['Method Description:\n\nWe used molecular dynamics with LAMMPS to equilibrate an Al sample under isobaric, isothermal conditions (NPT ensemble) at 300 K and 1 atm for 500 ps. The initial conditions were obtained by replicating the fcc unit cell 5 times in each direction. We used Nose-Hoover thermostat and barostat with relaxation timescales of 0.1 and 1 ps, respectively. All MD simulations were performed using LAMMPS. Atomic interactions were obtained using an embedded atom model developed by Ercolessi and Adams [1] obtained from OpenKIM.org. [1] EAM alloy potential for Al developed by Ercolessi and Adams (1994) v002. OpenKIM. 2018. doi:10.25950/376e3e7e.', 'Method Description:\n\nWe characterized the melting of a bulk Ni sample using molecular dynamics with LAMMPS. The initial condition was obtained by replicating the Ni u

In [58]:
prompt_dict = {'prompt1':prompts[0],'prompt2':prompts[1],'prompt3':prompts[2]}

model_dict = {model_dirs[0]:model_names[0],
              model_dirs[1]:model_names[1],
              model_dirs[2]:model_names[2],
              model_dirs[3]:model_names[3],
              model_dirs[4]:model_names[4],
             }
print(prompt_dict)


print('Total Prompt/Model Options:\n------------------------------------')
print(prompt_dirs)
print(model_dirs)
print(model_names)

# p_dir_sel = prompt_dirs[:1]
p_dir_sel = prompt_dirs[:]
m_dir_sel = list(model_dict.keys())[0:1] #[1:3]
m_name_sel = list(model_dict.values())[0:1] #[1:3]

print('Subset Prompt/Model Options:\n------------------------------------')
print(p_dir_sel)
print(m_dir_sel)
print(m_name_sel)

{'prompt1': 'Method Description:\n\nWe used molecular dynamics with LAMMPS to equilibrate an Al sample under isobaric, isothermal conditions (NPT ensemble) at 300 K and 1 atm for 500 ps. The initial conditions were obtained by replicating the fcc unit cell 5 times in each direction. We used Nose-Hoover thermostat and barostat with relaxation timescales of 0.1 and 1 ps, respectively. All MD simulations were performed using LAMMPS. Atomic interactions were obtained using an embedded atom model developed by Ercolessi and Adams [1] obtained from OpenKIM.org. [1] EAM alloy potential for Al developed by Ercolessi and Adams (1994) v002. OpenKIM. 2018. doi:10.25950/376e3e7e.', 'prompt2': 'Method Description:\n\nWe characterized the melting of a bulk Ni sample using molecular dynamics with LAMMPS. The initial condition was obtained by replicating the Ni unit cell 10 times in each direction. Initial velocities were drawn from the Maxwell-Boltzmann distribution at 600 K. The system was heated fro

# OpenAI API Script Generation

CAREFUL!!!! - Running this will overwrite current scripts

In [59]:
#OPENAI
for n, prompt_type in enumerate(p_dir_sel):
    for m, model_type in enumerate(m_dir_sel):
        for trial in np.arange(0,1,1):
            print('Trial: ',trial)
            print('Model: ',m_name_sel[m])
            response = client.chat.completions.create(
            model=m_name_sel[m],
            messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": prompt_dict[prompt_type]},
                ],
            )
            
            response_text = response.choices[0].message.content
            print('Prompt #:',prompt_type)
            print('------------------------------')
            print('Prompt:\n',prompt_dict[prompt_type])
            print('------------------------------')
            print('System Prompt:\n',SYSTEM_PROMPT)
            print('------------------------------')
            print(response_text)

#             text_to_file_basic(response_text, prompt_type, model_type, trial, scripts_dir) # for all others            
            text_to_file(response_text, prompt_type, model_type, trial, scripts_dir) # for gpt-4o

Trial:  0
Model:  gpt-4o-2024-08-06
Prompt #: prompt1
------------------------------
Prompt:
 Method Description:

We used molecular dynamics with LAMMPS to equilibrate an Al sample under isobaric, isothermal conditions (NPT ensemble) at 300 K and 1 atm for 500 ps. The initial conditions were obtained by replicating the fcc unit cell 5 times in each direction. We used Nose-Hoover thermostat and barostat with relaxation timescales of 0.1 and 1 ps, respectively. All MD simulations were performed using LAMMPS. Atomic interactions were obtained using an embedded atom model developed by Ercolessi and Adams [1] obtained from OpenKIM.org. [1] EAM alloy potential for Al developed by Ercolessi and Adams (1994) v002. OpenKIM. 2018. doi:10.25950/376e3e7e.
------------------------------
System Prompt:
 System Prompt:
You are an expert in molecular dynamics simulations and LAMMPS scripting. Your task is to generate complete, runnable LAMMPS input scripts based only on the provided method descriptio

## Anthropic Script Maker
Careful!!! - Running this will overwrite your scripts (well, 2 down will)

In [60]:
prompt_dict = {'prompt1':prompts[0],'prompt2':prompts[1],'prompt3':prompts[2]}

model_dict = {model_dirs[0]:model_names[0],
              model_dirs[1]:model_names[1],
              model_dirs[2]:model_names[2],
              model_dirs[3]:model_names[3],
              model_dirs[4]:model_names[4],
             }
# print(prompt_dict)


print('Total Prompt/Model Options:\n------------------------------------')
print(prompt_dirs)
print(model_dirs)
print(model_names)

# p_dir_sel = prompt_dirs[:1]
p_dir_sel = prompt_dirs[:]
m_dir_sel = list(model_dict.keys())[3:4] #[1:3]
m_name_sel = list(model_dict.values())[3:4] #[1:3]

print('Subset Prompt/Model Options:\n------------------------------------')
print(p_dir_sel)
print(m_dir_sel)
print(m_name_sel)

Total Prompt/Model Options:
------------------------------------
['prompt1', 'prompt2', 'prompt3']
['gpt-4o', 'gpt-4.1', 'gpt-o3', 'claude-opus-4', 'gpt-5']
['gpt-4o-2024-08-06', 'gpt-4.1-2025-04-14', 'o3-2025-04-16', 'claude-4-opus-20250514', 'gpt-5-2025-08-07']
Subset Prompt/Model Options:
------------------------------------
['prompt1', 'prompt2', 'prompt3']
['claude-opus-4']
['claude-4-opus-20250514']


In [61]:
# Just Anthropic
# print(model_type)
# model_type='claude-4-opus'
# print(prompt_dict)
for n, prompt_type in enumerate(prompt_dirs):
    for m, model_type in enumerate(m_name_sel):
        for trial in np.arange(0,1,1):
            print('Trial: ',trial)
            print('Model: ',m_name_sel[m])
            response = client2.messages.create(
                model=m_name_sel[m],
                max_tokens=1000,
                temperature=0.7,
                system=SYSTEM_PROMPT,
                messages=[
                    {
                        "role": "user",
                        "content": prompt_dict[prompt_type]
                    }
                ]
            )
            response_text = response.content[0].text
            
            print('Prompt #:',prompt_type)
            print('------------------------------')
            print('Prompt:\n',prompt_dict[prompt_type])
            print('------------------------------')
            print('System Prompt:\n',SYSTEM_PROMPT)
            print('------------------------------')
            print(response_text)
            text_to_file(response_text, prompt_type, model_type, trial, scripts_dir)

Trial:  0
Model:  claude-4-opus-20250514


NameError: name 'client2' is not defined